In [1]:
import pandas as pd

# Read the CSV file
df = pd.read_csv("scored_constituents_with_price_sample_7-2-2025.csv")

# Convert to JSON and save
df.to_json("scored_constituents_with_price_sample_7-2-2025.json", orient="records", indent=4)


In [20]:
# ──── Imports ────────────────────────────────────────────────────────────────
import threading, time, datetime, os
from ibapi.client import EClient
from ibapi.wrapper import EWrapper
from ibapi.contract import Contract
from ibapi.order import Order
from ibapi.execution import ExecutionFilter
import pandas as pd
import matplotlib.pyplot as plt
from fpdf import FPDF
import yfinance as yf

# ──── IB API Client Class ────────────────────────────────────────────────────
class IBApi(EWrapper, EClient):
    def __init__(self):
        EClient.__init__(self, self)
        self.nextOrderId = None
        self.connected_evt = threading.Event()
        self.account_values = []
        self.exec_details = []
    def nextValidId(self, orderId: int):
        self.nextOrderId = orderId
        self.connected_evt.set()
    def orderStatus(self, *args): pass
    def error(self, reqId, errorCode, errorString): print(f"[ERROR] {errorString}")
    def accountSummary(self, reqId, account, tag, value, currency):
        if tag == "NetLiquidation":
            ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            self.account_values.append((ts, float(value)))
    def accountSummaryEnd(self, reqId): pass
    def execDetails(self, reqId, contract, execution):
        self.exec_details.append({
            'symbol': contract.symbol,
            'action': execution.side,
            'qty': execution.shares,
            'price': execution.price,
            'time': execution.time
        })
    def execDetailsEnd(self, reqId): pass

# ──── Main Bot Class ─────────────────────────────────────────────────────────
class IBTradingBot:
    def __init__(self, usd_amount=150.0):
        self.usd_amount = usd_amount
        self.ib = IBApi()
        self.clientId = 1
        self.port = 7496
        self.host = "127.0.0.1"
        self.pos = 0
        self.ticker = None

    def connect(self, clientId=1):
        self.ib.connect(self.host, self.port, clientId)
        threading.Thread(target=self.ib.run, daemon=True).start()
        if not self.ib.connected_evt.wait(5):
            raise RuntimeError("No nextValidId")

    def disconnect(self):
        self.ib.disconnect()

    def make_contract(self, symbol):
        c = Contract()
        c.symbol, c.secType, c.exchange, c.primaryExchange, c.currency = symbol, "STK", "SMART", "ISLAND", "USD"
        return c

    def place_market_order(self, symbol, action):
        price = 100
        qty = max(int(self.usd_amount // price), 1)
        oid = self.ib.nextOrderId
        self.ib.nextOrderId += 1
        self.ib.placeOrder(oid, self.make_contract(symbol), Order(action=action, orderType="MKT", totalQuantity=qty, tif="DAY", transmit=True))
        self.pos = 1 if action == "BUY" else 0

    def buy(self, s): self.place_market_order(s, "BUY")
    def close_buy(self, s): self.place_market_order(s, "SELL")

    def request_account_value(self):
        self.ib.reqAccountSummary(9001, "All", "NetLiquidation")
        time.sleep(2)
        self.ib.cancelAccountSummary(9001)

    def request_executions(self):
        f = ExecutionFilter()
        f.time = (datetime.datetime.now() - datetime.timedelta(days=360)).strftime("%Y%m%d-%H:%M:%S")
        self.ib.reqExecutions(9002, f)
        time.sleep(2)

    def fetch_ticker_info(self, ticker):
        tk = yf.Ticker(ticker)
        info = tk.info
        return {
            "Price": info.get("regularMarketPrice", "N/A"),
            "Bid": info.get("bid", "N/A"),
            "Ask": info.get("ask", "N/A"),
            "EPS": info.get("trailingEps", "N/A"),
            "Market Cap": info.get("marketCap", "N/A"),
            "P/E": info.get("trailingPE", "N/A"),
            "Sector": info.get("sector", "N/A"),
            "Industry": info.get("industry", "N/A")
        }

    def generate_eod_report(self, filename="eod_report.pdf", ticker="AAPL"):
        self.request_account_value()
        self.request_executions()
        df = pd.DataFrame(self.ib.account_values, columns=["Timestamp","NetLiquidation"])
        df["Timestamp"] = pd.to_datetime(df["Timestamp"])
        df["Shift"] = df["NetLiquidation"].shift(1)
        df2 = df[df["NetLiquidation"] != df["Shift"]].copy()
        plt.figure(figsize=(10,4))
        if len(df2) > 1:
            plt.plot(df2["Timestamp"], df2["NetLiquidation"], marker="o", color="blue")
        else:
            val = df["NetLiquidation"].iloc[-1]
            plt.axhline(val, linestyle="--", color="gray")
        plt.title("Net Liquidation")
        plt.xlabel("Time"); plt.ylabel("USD"); plt.grid(True)
        plt.tight_layout()
        plt.savefig("equity.png"); plt.close()

        info = self.fetch_ticker_info(ticker)
        pdf = FPDF()
        pdf.add_page()
        pdf.set_font("Arial","B",16); pdf.cell(0,10,"EOD Report",ln=True)
        pdf.set_font("Arial","",12); pdf.cell(0,10,f"Generated: {datetime.datetime.now():%Y-%m-%d %H:%M:%S}",ln=True)
        pdf.ln(5)
        pdf.image("equity.png", w=120)

        # Ticker info on right
        pdf.set_xy(135, 30)
        pdf.set_font("Arial","B",12); pdf.cell(0,8,f"Ticker: {ticker}", ln=True)
        pdf.set_font("Arial","",11)
        for k,v in info.items():
            pdf.set_x(135); pdf.cell(0,6,f"{k}: {v}", ln=True)
        pdf.ln(10)

        # Executions table
        pdf.set_font("Arial","B",14); pdf.cell(0,10,"Executions :",ln=True)
        df_exec = pd.DataFrame(self.ib.exec_details)
        if "time" in df_exec.columns:
            df_exec["time"] = pd.to_datetime(df_exec["time"], errors="coerce")
            df_exec = df_exec.sort_values("time", ascending=False)
        pdf.set_font("Arial","",10)
        col_w = [20,20,20,25,60]
        for h in ["symbol","action","qty","price","time"]:
            pdf.cell(col_w[["symbol","action","qty","price","time"].index(h)], 8, h.title(), border=1)
        pdf.ln()
        for _,r in df_exec.iterrows():
            pdf.cell(col_w[0],8,r.symbol,border=1)
            pdf.cell(col_w[1],8,r.action,border=1)
            pdf.cell(col_w[2],8,str(r.qty),border=1)
            pdf.cell(col_w[3],8,f"{r.price:.2f}",border=1)
            pdf.cell(col_w[4],8,str(r.time)[:19],border=1)
            pdf.ln()

        pdf.output(filename)
        os.remove("equity.png")
        print(f"[PDF] Saved {filename}")

# ──── Entry Point ────────────────────────────────────────────────────────────
if __name__ == "__main__":
    ticker_input = input("Enter the ticker you focused on today (e.g. AAPL): ").upper()
    bot = IBTradingBot()
    bot.connect()
    bot.generate_eod_report("daily_report_with_ticker.pdf", ticker=ticker_input)
    bot.disconnect()


Enter the ticker you focused on today (e.g. AAPL):  CCL


[ERROR] Market data farm connection is OK:usfarm.nj
[ERROR] Market data farm connection is OK:eufarm
[ERROR] Market data farm connection is OK:cashfarm
[ERROR] Market data farm connection is OK:usfarm
[ERROR] HMDS data farm connection is OK:euhmds
[ERROR] HMDS data farm connection is OK:ushmds
[ERROR] Sec-def data farm connection is OK:secdefeu
[PDF] Saved daily_report_with_ticker.pdf
